# Energy Optimization — All-Scenarios Report Generator

Runs the full GEKKO MINLP + post-processing pipeline **once per scenario** and emits:

- `eo_report_<scenario>_<ts>.xlsx` and `.pdf` per scenario
- `eo_all_scenarios_<ts>.xlsx` — combined cross-scenario comparison (one row per scenario)

Edit `SCENARIOS_TO_RUN` in the config cell to control the set. Default: all feasible scenarios
(infeasibility test cases excluded).

**Wall time:** ~30 min × N scenarios. Default N = 13 → 6–7 hours.


In [ ]:
%pip install pandas --quiet
%pip install numpy --quiet
%pip install xlsxwriter --quiet
%pip install openpyxl --quiet
%pip install gekko --quiet
%pip install networkx --quiet
%pip install scipy --quiet
%pip install weasyprint --quiet


In [ ]:
# ── SCENARIO CONFIGS + RUN LIST ──────────────────────────────────────────────
_BASELINE = {'Power_Rate': 13.334, 'Fuel_Rate': 2.04, 'DMW_Rate': 7.34 / 3.75}
_ALL_TURBINES = [
    'BFW_B_Turb_Status', 'BFW_C_Turb_Status', 'BFW_E_Turb_Status',
    'VHP_BFW_B_Turb_Status', 'VHP_BFW_C_Turb_Status',
    'CW_Turbine_A_Status', 'CW_Turbine_B_Status', 'CW_Turbine_G_Status',
    'Air_Compressor_Turbine_A_Status', 'Air_Compressor_Turbine_D_Status',
    'DMW_Turbine_A_Status', 'DMW_Turbine_C_Status',
]

SCENARIO_CONFIGS = {
    'baseline':    {'prices': dict(_BASELINE), 'bound_patches': [],
                    'description': 'Baseline prices'},
    'power_low':   {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 0.85}, 'bound_patches': []},
    'power_high':  {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 1.15}, 'bound_patches': []},
    'power_vhigh': {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 1.30}, 'bound_patches': []},
    'power_vlow':  {'prices': {**_BASELINE, 'Power_Rate': _BASELINE['Power_Rate'] * 0.70}, 'bound_patches': []},
    'fuel_low':    {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 0.85}, 'bound_patches': []},
    'fuel_high':   {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 1.15}, 'bound_patches': []},
    'fuel_vhigh':  {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 1.30}, 'bound_patches': []},
    'fuel_vlow':   {'prices': {**_BASELINE, 'Fuel_Rate':  _BASELINE['Fuel_Rate']  * 0.70}, 'bound_patches': []},
    'dmw_low':     {'prices': {**_BASELINE, 'DMW_Rate':   _BASELINE['DMW_Rate']   * 0.85}, 'bound_patches': []},
    'dmw_high':    {'prices': {**_BASELINE, 'DMW_Rate':   _BASELINE['DMW_Rate']   * 1.15}, 'bound_patches': []},
    'fuel_10x': {
        'prices': {**_BASELINE, 'Fuel_Rate': _BASELINE['Fuel_Rate'] * 10.0},
        'bound_patches': [],
        'description': 'Stress: Fuel at 10x baseline'
    },
    'negative_power': {
        'prices': {**_BASELINE, 'Power_Rate': -5.0},
        'bound_patches': [],
        'description': 'Stress: Grid pays plant to consume electricity'
    },
    # ── Infeasibility tests (excluded by default) ────────────────────────────
    'infeas_3_boilers': {
        'prices': dict(_BASELINE),
        'bound_patches': [
            {'tag_name': 'BLR_1_Status', 'lower': 0.0, 'upper': 0.0},
            {'tag_name': 'BLR_2_Status', 'lower': 0.0, 'upper': 0.0},
            {'tag_name': 'BLR_3_Status', 'lower': 0.0, 'upper': 0.0},
        ],
        'description': 'Infeasibility: BLR_1/2/3 forced offline'
    },
    'infeas_all_turbines': {
        'prices': dict(_BASELINE),
        'bound_patches': [{'tag_name': t, 'lower': 0.0, 'upper': 0.0} for t in _ALL_TURBINES],
        'description': 'Infeasibility: all 12 steam turbines OFF'
    },
    'infeas_1_boiler': {
        'prices': dict(_BASELINE),
        'bound_patches': [{'tag_name': 'Total_Boilers_Running', 'lower': 1.0, 'upper': 1.0}],
        'description': 'Infeasibility: pin to single boiler'
    },
}

# Default: 13 feasible scenarios (drop infeasibility tests).
# To include them, add their names back to this list.
SCENARIOS_TO_RUN = [
    'baseline',
    'power_low', 'power_high', 'power_vhigh', 'power_vlow',
    'fuel_low',  'fuel_high',  'fuel_vhigh',  'fuel_vlow',
    'dmw_low',   'dmw_high',
    'fuel_10x',  'negative_power',
]

# Per-scenario sweep is OFF by default (slow); flip to True if needed
RUN_SENSITIVITY_SWEEP = False

# Fail-fast or continue-on-error
CONTINUE_ON_ERROR = True

# Output directory for per-scenario reports
REPORTS_DIR_NAME = 'scenario_reports'

print(f'Scenarios to run: {len(SCENARIOS_TO_RUN)}')
for s in SCENARIOS_TO_RUN: print(f'  - {s}')


In [ ]:
# ── SETUP — imports, formula engine, constants ─────────────────────────────
import re, math, warnings, os, sys, time, shutil
import numpy as np
import pandas as pd
import networkx as nx
from datetime import datetime
from pathlib import Path
from openpyxl import load_workbook

try:
    from scipy.optimize import fsolve
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False

warnings.filterwarnings('ignore')
try: sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception: pass

ROOT         = Path('.').resolve()
FEATURE_FILE = str(ROOT / 'feature_file_eo_v7_unified.xlsx')
OUTPUT_DIR   = ROOT / 'output'
DB_DIR       = ROOT / 'tables_from_db'
OUTPUT_DIR.mkdir(exist_ok=True)

PI_ROW_SELECTION = 'best'
MODEL_ID  = 1
CASE_ID   = 1
TARGET_TS = '2026-03-31 00:00:00.000'

BASELINE = {'Power_Rate': 13.334, 'Fuel_Rate': 2.04, 'DMW_Rate': 7.34/3.75}
FUEL_COST_IN_MMBTU_BASELINE = 2.15

IMBALANCE_PARAMS = {
    'HP_Steam_Imbalance','MP_Steam_Imbalance','LP_Steam_Imbalance',
    'Total_Fuel_Imbalance','BFW_Imbalance_Enthalpy',
    'BFW_A_Motor_Power_Imbalance','BFW_B_Turb_Steam_Imbalance',
    'BFW_C_Turb_Steam_Imbalance','BFW_D_Motor_Power_Imbalance',
    'BFW_E_Turb_Steam_Imbalance','BFW_F_Motor_Power_Imbalance',
    'DMW_Turbine_A_Steam_Imbalance','DMW_Turbine_C_Steam_Imbalance',
    'DMW_Pump_Motor_B_Power_Imbalance',
    'Air_Compressor_Turbine_A_Steam_Imbalance','Air_Compressor_Turbine_D_Steam_Imbalance',
    'Air_Compressor_Motor_C_Power_Imbalance','Air_Compressor_Motor_B_Power_Imbalance',
}
SKIP_CONSTRAINTS = set()

# ── Formula engine (verbatim from Optimizer_MINLP Cell 1) ──────────────────
TAG_PATTERN = re.compile(r'\[([^\[\]]+)\]')

def extract_tag_refs(formula):
    if not isinstance(formula, str): return []
    return [t.strip().replace(' ','_') for t in TAG_PATTERN.findall(formula)]

def preprocess_formula(formula):
    if not isinstance(formula, str): return str(formula)
    formula = formula.replace('(Total_Fuel_Consumption_U_O)','([Total_Fuel_Consumption_U_O])')
    s = TAG_PATTERN.sub(lambda m: m.group(1).strip().replace(' ','_'), formula)
    s = s.replace('^','**').replace('&&',' and ').replace('||',' or ')
    s = re.sub(r'(?i)\bif\s*\(', 'if_(', s)
    return s

def _make_eval_env(context, all_tag_names=None):
    def _if_(c, t, f):
        try:
            cond = float(c)
            if np.isnan(cond): cond = 0.0
        except: cond = bool(c)
        return t if cond else f
    def _min_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],'__iter__') else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return min(v) if v else np.nan
    def _max_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],'__iter__') else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return max(v) if v else np.nan
    def _avg_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],'__iter__') else list(a)
        v = [float(x) for x in args if not (isinstance(x,float) and np.isnan(x))]
        return sum(v)/len(v) if v else np.nan
    def _sum_(*a):
        args = list(a[0]) if len(a)==1 and hasattr(a[0],'__iter__') else list(a)
        v = [float(x) for x in args if x is not None and not (isinstance(x,float) and np.isnan(x))]
        return sum(v) if v else 0.0
    env = {'__builtins__':{},'nan':np.nan,'inf':np.inf,'pi':math.pi,
           'if_':_if_,'if':_if_,'If':_if_,
           'min':_min_,'max':_max_,'avg':_avg_,'sum':_sum_,
           'abs':abs,'round':round,
           'sqrt': lambda x: math.sqrt(max(float(x),0)) if isinstance(x,(int,float,np.integer,np.floating)) else np.nan,
           'log':  lambda x: math.log(max(float(x),1e-30)) if isinstance(x,(int,float,np.integer,np.floating)) else np.nan,
           'ln':   lambda x: math.log(max(float(x),1e-30)) if isinstance(x,(int,float,np.integer,np.floating)) else np.nan,
           'exp':  lambda x: math.exp(min(float(x),700)),
           'ceil':math.ceil,'floor':math.floor,'trunc':math.trunc,
           'sin':math.sin,'cos':math.cos,
           'missing': lambda x: 1.0 if (x is None or (isinstance(x,float) and np.isnan(x))) else 0.0,
           'MISSING_NUMERIC':float('nan'),
           '_safe_div': lambda a,b: float('nan') if (b==0 or b is None) else a/b,
           'True':True,'False':False}
    if all_tag_names:
        for tn in all_tag_names:
            if tn not in env: env[tn] = np.nan
    env.update({k:v for k,v in context.items() if not (isinstance(v,str) or callable(v))})
    return env

def safe_eval_scalar(formula, context, all_tags=None):
    try:
        return float(eval(preprocess_formula(formula), _make_eval_env(context, all_tags)))
    except Exception:
        return np.nan

def topo_sort_formulas(formula_map):
    G = nx.DiGraph()
    all_tags = set(formula_map.keys())
    for tag, formula in formula_map.items():
        G.add_node(tag)
        for ref in extract_tag_refs(formula):
            if ref in all_tags and ref != tag:
                G.add_edge(ref, tag)
    cycles = list(nx.simple_cycles(G))
    for cycle in cycles:
        for i in range(len(cycle)):
            try: G.remove_edge(cycle[i], cycle[(i+1)%len(cycle)])
            except nx.NetworkXError: pass
    return list(nx.topological_sort(G)), cycles

def _safe_float(v, default=0.0):
    if v is None: return default
    try:
        f = float(v)
        return default if f != f else f
    except Exception:
        return default

def gval(v):
    try:
        raw = v.value
        return float(list(raw)[0] if hasattr(raw,'__iter__') else raw)
    except Exception:
        return np.nan

print('Setup complete. ROOT:', ROOT)


In [ ]:
# ── DATA INGESTION — read all sheets from feature file ─────────────────────
xl = pd.ExcelFile(FEATURE_FILE)
_OPTIMIZER_SHEETS = ['tag','inferred','variables','derived_equations',
                     'constraints','objective','model_parameter','master_pi_data']
_POST_SHEETS = ['seu_detail','peeo_based_adjustment','cause','effect','ods',
                'output_pi_mapping','derived_equation_post_optimizer',
                'inferred_tag_rm_block_mapping','seu_suggestions_mapping']

cfg = {}
for sheet in _OPTIMIZER_SHEETS + _POST_SHEETS:
    if sheet in xl.sheet_names:
        cfg[sheet] = xl.parse(sheet).dropna(how='all').reset_index(drop=True)
        print(f'  {sheet:<35}: {len(cfg[sheet]):>5} rows')
    else:
        cfg[sheet] = pd.DataFrame()
        print(f'  {sheet:<35}: MISSING')

model_params = {}
for _, row in cfg['model_parameter'].iterrows():
    p = str(row.get('parameter','')).strip()
    v = row.get('value', np.nan)
    if p and p != 'nan' and pd.notna(v):
        try: model_params[p] = float(v)
        except: pass
print(f'\n  Model parameters: {len(model_params)}')

# ── DB CSV tables (graceful fallback to empty DataFrame) ───────────────────
_DB_TABLES = ['seu_details','seec_kpi','tag','cause','effect',
              'operation_decision_support','peeo_ods_info',
              'pi_seu_tag_mapping','switch_configuration','message_info']
db = {}
for t in _DB_TABLES:
    p = DB_DIR / f'{t}.csv'
    if p.exists():
        try:
            db[t] = pd.read_csv(str(p), encoding='utf-8', low_memory=False)
        except Exception:
            db[t] = pd.DataFrame()
    else:
        db[t] = pd.DataFrame()
n_db = sum(1 for v in db.values() if not v.empty)
print(f'  DB CSV tables loaded: {n_db}/{len(_DB_TABLES)}')

# ── Build inferred formula map ─────────────────────────────────────────────
inferred_formula_map = {}
for _, row in cfg['inferred'].iterrows():
    tag = str(row.get('tag_name','')).strip()
    f   = row.get('formula_expression','')
    if tag and tag != 'nan':
        inferred_formula_map[tag] = str(f) if pd.notna(f) else ''
if 'Fuel_Bill' in inferred_formula_map:
    inferred_formula_map['Fuel_Bill'] = inferred_formula_map['Fuel_Bill'].replace(
        '(Total_Fuel_Consumption_U_O)','([Total_Fuel_Consumption_U_O])')

# collect all referenced tag names for the eval environment
all_referenced_tags = set()
for f in inferred_formula_map.values(): all_referenced_tags.update(extract_tag_refs(f))
for _, r in cfg['derived_equations'].iterrows():
    f = r.get('formula_expression','')
    if isinstance(f, str): all_referenced_tags.update(extract_tag_refs(f))
for _, r in cfg['constraints'].iterrows():
    f = r.get('expression','')
    if isinstance(f, str): all_referenced_tags.update(extract_tag_refs(f))
all_referenced_tags.update(inferred_formula_map.keys())
df_pi = cfg['master_pi_data'].copy()
ts_col = next((c for c in df_pi.columns if 'date' in c.lower()), None)
all_referenced_tags.update(c for c in df_pi.columns if c != ts_col)

sorted_inf, circular_inf = topo_sort_formulas(inferred_formula_map)
if circular_inf: print(f'  Circular deps removed: {len(circular_inf)}')
print(f'  Inferred formulas: {len(inferred_formula_map)}   Referenced tags: {len(all_referenced_tags)}')


In [ ]:
# ── PER-SCENARIO RUNNER ──────────────────────────────────────────────────────
# Wraps the full optimizer + post-process + per-scenario report write.
# Returns a summary dict consumed by the loop cell to build the combined report.

def run_scenario(SCENARIO):
    """Run one scenario end-to-end. Writes per-scenario xlsx+pdf. Returns summary dict."""
    if SCENARIO not in SCENARIO_CONFIGS:
        raise ValueError(f'Unknown SCENARIO: {SCENARIO!r}')

    _cfg = SCENARIO_CONFIGS[SCENARIO]
    Power_Rate    = _cfg['prices']['Power_Rate']
    Fuel_Rate     = _cfg['prices']['Fuel_Rate']
    DMW_Rate      = _cfg['prices']['DMW_Rate']
    bound_patches = list(_cfg.get('bound_patches', []))

    print(f'\n{"#"*70}\n# SCENARIO: {SCENARIO}')
    print(f'#   Power={Power_Rate:.4f}  Fuel={Fuel_Rate:.4f}  DMW={DMW_Rate:.4f}')
    if bound_patches: print(f'#   Bound patches: {len(bound_patches)}')
    print(f'{"#"*70}')

    # ── PI SNAPSHOT → INFERRED CHAIN → APPLY SCENARIO → VAR BOUNDS ────────────
    # ── Step 1: PI snapshot ────────────────────────────────────────────────────
    if ts_col: df_pi[ts_col] = pd.to_datetime(df_pi[ts_col], errors='coerce')
    if PI_ROW_SELECTION == 'best':
        snapshot_idx = df_pi.notna().sum(axis=1).idxmax()
    elif PI_ROW_SELECTION == 'first': snapshot_idx = 0
    elif PI_ROW_SELECTION == 'last':  snapshot_idx = len(df_pi) - 1
    else: snapshot_idx = int(PI_ROW_SELECTION)
    pi_snapshot = df_pi.iloc[[snapshot_idx]].reset_index(drop=True)
    ts_value = pi_snapshot[ts_col].iloc[0] if ts_col else 'N/A'

    working_ctx = {}
    for col in pi_snapshot.columns:
        if ts_col and col == ts_col: continue
        v = pi_snapshot.iloc[0][col]
        if pd.notna(v):
            try: working_ctx[col] = float(v)
            except: pass
    working_ctx.update(model_params)
    working_ctx.setdefault('Whatif_running', 0.0)
    working_ctx.setdefault('opt_flag', 0.0)
    working_ctx.setdefault('act_running', 1.0)
    PI_ALIAS = {'AC_A_Flow_To_Header_raw':'Air_compressor_A_Discharge_flow_raw',
                'AC_B_Flow_To_Header_raw':'Air_compressor_B_Discharge_flow_raw',
                'AC_C_Flow_To_Header_raw':'Air_compressor_C_Discharge_flow_raw',
                'AC_D_Flow_To_Header_raw':'Air_compressor_D_Discharge_flow_raw'}
    for a, rc in PI_ALIAS.items():
        if a not in working_ctx and rc in working_ctx: working_ctx[a] = working_ctx[rc]
    lhv = working_ctx.get('LHV_Raw', np.nan)
    if not (isinstance(lhv, float) and np.isnan(lhv)):
        for t in ['LHV_Raw_2','LHV_Raw_3','LHV_Raw_4','LHV_Raw_5']:
            working_ctx.setdefault(t, lhv)
    print(f'  Snapshot: {ts_value} (row {snapshot_idx})   PI values: {len(working_ctx)}')

    # ── Step 2: Baseline inferred chain ───────────────────────────────────────
    n_computed = n_skipped = 0
    for tag in sorted_inf:
        f = inferred_formula_map.get(tag,'')
        if not f.strip(): continue
        v = safe_eval_scalar(f, working_ctx, all_referenced_tags)
        if not np.isnan(v): working_ctx[tag] = v; n_computed += 1
        else: n_skipped += 1
    print(f'  Inferred: computed={n_computed}  skipped(NaN)={n_skipped}')

    # ── Step 3: Apply scenario prices (direct override — no Excel write) ───────
    prices = {'Power_Rate': float(Power_Rate), 'Fuel_Rate': float(Fuel_Rate), 'DMW_Rate': float(DMW_Rate)}
    working_ctx.update(prices)
    working_ctx['Fuel_Cost_in_MMBTU'] = prices['Fuel_Rate'] * FUEL_COST_IN_MMBTU_BASELINE / BASELINE['Fuel_Rate']

    # Re-evaluate inferred chain with new prices so downstream formulas update
    for tag in sorted_inf:
        f = inferred_formula_map.get(tag,'')
        if not f.strip(): continue
        v = safe_eval_scalar(f, working_ctx, all_referenced_tags)
        if not np.isnan(v): working_ctx[tag] = v
    # Re-apply prices once more (chain formula may have reset them to baseline value)
    working_ctx.update(prices)
    working_ctx['Fuel_Cost_in_MMBTU'] = prices['Fuel_Rate'] * FUEL_COST_IN_MMBTU_BASELINE / BASELINE['Fuel_Rate']

    print(f'  Prices applied — Power={prices["Power_Rate"]:.4f} $/MWh | Fuel={prices["Fuel_Rate"]:.4f} $/MMBTU | DMW={prices["DMW_Rate"]:.4f} $/t')

    # ── Step 4: Build actual_ns ────────────────────────────────────────────────
    pi_tag_names = set(cfg['tag'][cfg['tag']['tag_type']=='pi']['tag_name'].tolist()) if 'tag_type' in cfg['tag'].columns else set()
    actual_ns = dict(working_ctx)
    for tag in pi_tag_names:
        if tag in working_ctx: actual_ns[f'{tag}_actual'] = working_ctx[tag]
    for tag in inferred_formula_map:
        if tag in working_ctx: actual_ns[f'{tag}_actual'] = working_ctx[tag]

    # ── Step 5: Variable bounds ────────────────────────────────────────────────
    env_pure = _make_eval_env(actual_ns, all_referenced_tags)
    def evaluate_dynamic_bound(expr_raw, default_val=None):
        if pd.isna(expr_raw) or str(expr_raw).strip() in ('','nan'): return default_val
        expr_str = str(expr_raw).strip()
        try: return float(expr_str)
        except ValueError:
            try: return float(eval(preprocess_formula(expr_str), env_pure))
            except Exception: return default_val

    imbalance_param_vals = {}
    var_defs = {}
    n_missing_pi = 0
    for _, row in cfg['variables'].iterrows():
        vname = str(row.get('tag_name','')).strip()
        if not vname or vname == 'nan': continue
        if vname in IMBALANCE_PARAMS:
            v = actual_ns.get(vname, np.nan)
            imbalance_param_vals[vname] = float(v) if not (isinstance(v,float) and np.isnan(v)) else 0.0
            continue
        lb_phys = float(row['lower_bound_value']) if pd.notna(row.get('lower_bound_value')) else 0.0
        ub_phys = float(row['upper_bound_value']) if pd.notna(row.get('upper_bound_value')) else max((lb_phys or 0)*2, 1e6)
        if lb_phys > ub_phys: lb_phys, ub_phys = ub_phys, lb_phys
        lb_scen = evaluate_dynamic_bound(row.get('lower_bound_expression'), lb_phys)
        ub_scen = evaluate_dynamic_bound(row.get('upper_bound_expression'), ub_phys)
        if lb_scen is None: lb_scen = lb_phys
        if ub_scen is None: ub_scen = ub_phys
        if lb_scen > ub_scen: lb_scen, ub_scen = ub_scen, lb_scen
        raw_val = actual_ns.get(vname, np.nan)
        if isinstance(raw_val,(int,float)) and np.isfinite(raw_val):
            init = float(raw_val); init_orig = float(raw_val)
        else:
            init = float(lb_phys); init_orig = float('nan'); n_missing_pi += 1
        var_defs[vname] = {'lb_phys':lb_phys,'ub_phys':ub_phys,'lb_scen':lb_scen,'ub_scen':ub_scen,
                           'init':init,'is_integer':bool(row.get('flag_integer',0)==1),'init_orig':init_orig}

    # FIX-1: Integer flag sanitation
    _INT_ALLOW_NAMES = {
        'BFW_Drives_Running','BFW_Motors_Running','BFW_Turbines_Running',
        'VHP_BFW_Drives_Running','VHP_BFW_Motors_Running','VHP_BFW_Turbines_Running',
        'CW_Motors_Running','CW_Turbines_Running','CW_Drives_Running',
        'DMW_Drives_Running','DMW_Motors_Running','DMW_Turbines_Running',
        'Air_Compressor_Drives_Running','Air_Compressor_Motors_Running','Air_Compressor_Turbines_Running',
        'Total_Boilers_Running',
    }
    for _vn, _vd in var_defs.items():
        should_be_int = _vn.endswith('_Status') or _vn in _INT_ALLOW_NAMES
        _vd['is_integer'] = should_be_int
        if should_be_int and _vn.endswith('_Status'):
            _vd['ub_phys'] = 1.0; _vd['ub_scen'] = 1.0
            _vd['lb_phys'] = 0.0; _vd['lb_scen'] = 0.0
    for _vn in _INT_ALLOW_NAMES:
        if _vn in var_defs:
            var_defs[_vn]['lb_phys'] = 0.0
            if var_defs[_vn]['lb_scen'] > 0: var_defs[_vn]['lb_scen'] = 0.0

    # ── Step 6: Apply bound patches (direct override — no Excel write) ─────────
    if bound_patches:
        for patch in bound_patches:
            tag = patch['tag_name']
            if tag in var_defs:
                if patch.get('lower') is not None:
                    var_defs[tag]['lb_scen'] = float(patch['lower'])
                    var_defs[tag]['lb_phys'] = float(patch['lower'])
                if patch.get('upper') is not None:
                    var_defs[tag]['ub_scen'] = float(patch['upper'])
                    var_defs[tag]['ub_phys'] = float(patch['upper'])
        print(f'  Bound patches applied: {len(bound_patches)}')

    n_int = sum(1 for v in var_defs.values() if v['is_integer'])
    print(f'  Decision variables: {len(var_defs)}  ({n_int} integer)  Missing PI: {n_missing_pi}')

    # ── GEKKO MODEL + 3-STAGE MINLP SOLVE ─────────────────────────────────────
    from gekko import GEKKO

    m = GEKKO(remote=False)
    m.options.IMODE = 3
    m.options.MAX_ITER = int(model_params.get('nlp_maximum_iterations', 1000))

    # Variables (Built using wide PHYSICAL bounds for Stage 1)
    gekko_vars = {}
    for vname, vd in var_defs.items():
        lb_g = int(round(vd['lb_phys'])) if vd['is_integer'] else vd['lb_phys']
        ub_g = int(round(vd['ub_phys'])) if vd['is_integer'] else vd['ub_phys']
        gekko_vars[vname] = m.Var(value=vd['init'], lb=lb_g, ub=ub_g,
                                  integer=vd['is_integer'], name=vname)
    n_int = sum(1 for v in var_defs.values() if v['is_integer'])
    print(f'  Decision Variables: {len(gekko_vars)} ({n_int} integer, {len(var_defs)-n_int} continuous)')

    # Imbalance Parameters (frozen at PI values)
    gekko_params = {}
    for pname, pval in imbalance_param_vals.items():
        gekko_params[pname] = m.Param(value=pval, name=pname)
    print(f'  Imbalance Parameters: {len(gekko_params)}')

    # Constraint / derived-equation skip sets (populated by SETUP; empty = enforce all)
    SKIP_CONSTRAINTS       = set()   # row indices in cfg['constraints'] to skip
    SKIP_DERIVED_IMBALANCE = set()   # tag names to skip in derived-equation loop

    def _strip_parens(s):
        s = s.strip()
        while s.startswith('(') and s.endswith(')'):
            d = 0; all_inner = True
            for i, ch in enumerate(s):
                if ch == '(': d += 1
                elif ch == ')': d -= 1
                if d == 0 and i < len(s) - 1: all_inner = False; break
            if all_inner: s = s[1:-1].strip()
            else: break
        return s

    def _is_scalar(x): return isinstance(x, (int, float, np.integer, np.floating))

    def make_gekko_env(ctx, m_obj):
        def _if_(c, t, f):
            try: return t if float(c) else f
            except: return m_obj.if3(c, t, f)
        def _min_(*args):
            args = list(args[0]) if len(args)==1 and hasattr(args[0], '__iter__') else list(args)
            try: return min(float(a) for a in args)
            except:
                r = args[0]
                for a in args[1:]: r = m_obj.min2(r, a)
                return r
        def _max_(*args):
            args = list(args[0]) if len(args)==1 and hasattr(args[0], '__iter__') else list(args)
            try: return max(float(a) for a in args)
            except:
                r = args[0]
                for a in args[1:]: r = m_obj.max2(r, a)
                return r
        env = {'__builtins__': {}, 'nan': 0.0, 'inf': 1e30, 'pi': math.pi,
               'if_': _if_, 'if': _if_, 'If': _if_, 'min': _min_, 'max': _max_,
               'avg': lambda *a: sum(float(x) for x in (a[0] if len(a)==1 else a))/len(a[0] if len(a)==1 else a),
               'abs':   lambda x: m_obj.abs(x)  if not _is_scalar(x) else abs(x),
               'sqrt':  lambda x: m_obj.sqrt(x) if not _is_scalar(x) else math.sqrt(max(float(x), 0)),
               'log':   lambda x: m_obj.log(x)  if not _is_scalar(x) else math.log(max(float(x), 1e-30)),
               'ln':    lambda x: m_obj.log(x)  if not _is_scalar(x) else math.log(max(float(x), 1e-30)),
               'exp':   lambda x: m_obj.exp(x)  if not _is_scalar(x) else math.exp(min(float(x), 700)),
               'ceil':  lambda x: math.ceil(float(x))  if _is_scalar(x) else x,
               'floor': lambda x: math.floor(float(x)) if _is_scalar(x) else x,
               'trunc': lambda x: math.trunc(float(x)) if _is_scalar(x) else x,
               'round': round,
               'sum':   lambda *a: sum(float(x) for x in (a[0] if len(a)==1 else a)),
               'sin': math.sin, 'cos': math.cos, 'missing': lambda x: 0.0,
               'MISSING_NUMERIC': float('nan'),
               '_safe_div': (lambda a, b: float('nan') if (b == 0 or b is None) else a / b),
               'True': True, 'False': False, 'm': m_obj}
        for t in all_referenced_tags:
            if t not in env: env[t] = 0.0
        env.update(ctx)
        return env

    gekko_ctx = {}
    for tag, val in actual_ns.items():
        if tag.endswith('_actual'): continue
        gekko_ctx[tag] = 0.0 if (isinstance(val, float) and np.isnan(val)) else val
    gekko_ctx.update(gekko_vars)
    gekko_ctx.update(gekko_params)
    gekko_env = make_gekko_env(gekko_ctx, m)

    # Derived equations → m.Intermediate (or m.Equation if tag is also a Var)
    derived_eq_map = {}
    for _, row in cfg['derived_equations'].iterrows():
        tag = str(row.get('tag_name', '')).strip()
        frm = row.get('formula_expression', '')
        if tag and tag != 'nan' and isinstance(frm, str) and frm.strip():
            derived_eq_map[tag] = frm
    sorted_derived, _ = topo_sort_formulas(derived_eq_map)

    gekko_interm = {}; failed_derived = []; n_eq_constraints = 0; n_skipped_derived = 0
    for tag in sorted_derived:
        formula = derived_eq_map.get(tag, '')
        if not formula: continue
        if tag in imbalance_param_vals or tag in SKIP_DERIVED_IMBALANCE:
            n_skipped_derived += 1
            continue
        try:
            result = eval(preprocess_formula(formula), gekko_env)
            if tag in gekko_vars:
                m.Equation(gekko_vars[tag] == result); iv = gekko_vars[tag]; n_eq_constraints += 1
            else:
                iv = m.Intermediate(result, name=tag)
            gekko_interm[tag] = iv
            gekko_env[tag] = iv; gekko_ctx[tag] = iv
        except Exception as e:
            failed_derived.append((tag, str(e)))
    print(f'  Intermediates: {len(gekko_interm)} ({n_eq_constraints} tied to DVs)  '
          f'Skipped (param): {n_skipped_derived}  Failed: {len(failed_derived)}')

    # Constraints
    n_constraints = 0; failed_constraints = []
    for ci, row in cfg['constraints'].iterrows():
        if ci in SKIP_CONSTRAINTS: continue
        raw = str(row.get('expression', '')).strip()
        if not raw or raw == 'nan': continue
        raw = _strip_parens(raw)
        prep = preprocess_formula(raw)
        try:
            for op in ['==', '>=', '<=', '>', '<']:
                if op in prep:
                    lhs_s, rhs_s = prep.split(op, 1)
                    lhs = eval(lhs_s.strip(), gekko_env)
                    rhs = eval(rhs_s.strip(), gekko_env)
                    if op == '==':         m.Equation(lhs == rhs)
                    elif op in ('>=', '>'): m.Equation(lhs >= rhs)
                    else:                   m.Equation(lhs <= rhs)
                    n_constraints += 1; break
        except Exception as e:
            failed_constraints.append((raw[:60], str(e)))
    print(f'  Constraints: {n_constraints}  Failed: {len(failed_constraints)}')

    # [FIX-2] Physics-repair constraints
    _physics_added = 0
    try:
        _dmw_sum = (gekko_vars['DMW_Turbine_A_Status']
                  + gekko_vars['DMW_Pump_Motor_B_Status']
                  + gekko_vars['DMW_Turbine_C_Status'])
        _rate = float(gekko_env.get('DMW_Pump_Rated_Flow', 350.0))
        if 'DMW_Makeup' in gekko_vars:
            m.Equation(gekko_vars['DMW_Makeup'] <= _dmw_sum * _rate); _physics_added += 1
    except Exception as _e: print(f'  [FIX-2] DMW link skipped: {_e}')
    try:
        _bfw_sum = sum(gekko_vars[n] for n in [
            'BFW_A_Motor_Status', 'BFW_B_Turb_Status', 'BFW_C_Turb_Status',
            'BFW_D_Motor_Status', 'BFW_E_Turb_Status', 'BFW_F_Motor_Status'] if n in gekko_vars)
        if not isinstance(_bfw_sum, int):
            m.Equation(_bfw_sum >= 1); _physics_added += 1
    except Exception as _e: print(f'  [FIX-2] BFW link skipped: {_e}')
    print(f'  [FIX-2] Physics-repair constraints added: {_physics_added}')

    # [FIX-3] Lock CW statuses at PI actual values
    _cw_locked = 0
    for _n in ['CW_Motor_D_Status', 'CW_Motor_E_Status', 'CW_Motor_F_Status',
               'CW_Turbine_A_Status', 'CW_Turbine_B_Status', 'CW_Turbine_G_Status']:
        if _n in gekko_vars and _n in var_defs:
            _act = 1
            var_defs[_n]['lb_phys'] = _act; var_defs[_n]['ub_phys'] = _act
            var_defs[_n]['lb_scen'] = _act; var_defs[_n]['ub_scen'] = _act
            var_defs[_n]['init']    = _act
            try:
                gekko_vars[_n].lower = _act; gekko_vars[_n].upper = _act; _cw_locked += 1
            except Exception as _e: print(f'  [FIX-3] CW lock skipped {_n}: {_e}')
    print(f'  CW status-lock constraints applied: {_cw_locked}')

    # [FIX-4a] Air-compressor discharge upper bounds
    _ac_pairs = [
        ('Air_compressor_A_Discharge_flow', 'Air_Compressor_Turbine_A_Status', 16.29),
        ('Air_compressor_B_Discharge_flow', 'Air_Compressor_Motor_B_Status',   16.29),
        ('Air_compressor_C_Discharge_flow', 'Air_Compressor_Motor_C_Status',   11.89),
        ('Air_compressor_D_Discharge_flow', 'Air_Compressor_Turbine_D_Status', 16.29),
    ]
    _ac_added = 0
    for _flow, _stat, _rmax in _ac_pairs:
        if _flow in gekko_vars and _stat in gekko_vars:
            try: m.Equation(gekko_vars[_flow] <= gekko_vars[_stat] * float(_rmax)); _ac_added += 1
            except Exception as _e: print(f'  [FIX-4a] skipped {_flow}: {_e}')
    print(f'  [FIX-4a] Air-comp discharge upper-bounds added: {_ac_added}')

    # Production lock: Boiler count
    try:
        actual_boilers = sum(round(float(actual_ns.get(f'BLR_{i}_Status', 0))) for i in range(1, 6))
        blr_status_vars = [gekko_vars[f'BLR_{i}_Status'] for i in range(1, 6)
                           if f'BLR_{i}_Status' in gekko_vars]
        if len(blr_status_vars) == 5:
            m.Equation(m.sum(blr_status_vars) == actual_boilers)
            print(f'  Boiler lock: {actual_boilers} boilers running')
        else:
            print(f'  Boiler lock WARNING: only found {len(blr_status_vars)}/5 boiler vars')
    except Exception as e:
        print(f'  Boiler lock skipped: {e}')

    # Objective
    obj_row     = cfg['objective'].iloc[0]
    obj_tag     = str(obj_row['tag_name']).strip()
    obj_dir     = float(obj_row['direction'])
    obj_formula = inferred_formula_map.get(obj_tag, '')
    obj_expr    = None
    if obj_formula:
        try:
            obj_expr = eval(preprocess_formula(obj_formula), gekko_env)
            is_sym = not isinstance(obj_expr, (int, float, np.integer, np.floating))
            print(f'  Objective evaluated: symbolic={is_sym}')
            if not is_sym:
                print(f'  WARNING: Objective is constant {obj_expr} — solver cannot optimize!')
        except Exception as e:
            print(f'  WARNING: Objective eval failed: {e}')
    if obj_expr is not None:
        if obj_dir == -1: m.Minimize(obj_expr)
        else:             m.Maximize(obj_expr)
        print(f'  Objective: {obj_tag} ({"minimize" if obj_dir==-1 else "maximize"})')

    # ── STAGE 1: Base reconciliation (IPOPT) ───────────────────────────────────
    print('\n──── STAGE 1: Base reconciliation (IPOPT) ────')
    m.options.SOLVER = 3
    m.options.MAX_ITER = 500
    t_solve = time.time()
    try:
        m.solve(disp=False)
        print('  Stage 1 OK: stable base balance found.')
    except Exception as e:
        print(f'  Stage 1 WARNING: {e}')

    # ── STAGE 2: Scenario shift (IPOPT) ────────────────────────────────────────
    print('\n──── STAGE 2: Scenario shift (IPOPT) ────')
    n_tightened = 0; stage2_prev_bounds = {}
    for vname, vd in var_defs.items():
        if vname not in gekko_vars or vd['is_integer']: continue
        if abs(vd['lb_scen'] - vd['lb_phys']) > 1e-4 or abs(vd['ub_scen'] - vd['ub_phys']) > 1e-4:
            stage2_prev_bounds[vname] = (gekko_vars[vname].lower, gekko_vars[vname].upper)
            gekko_vars[vname].lower = vd['lb_scen']; gekko_vars[vname].upper = vd['ub_scen']
            n_tightened += 1
    stage2_ok = False
    try:
        m.solve(disp=False); stage2_ok = True
        print(f'  Stage 2 OK: {n_tightened} vars tightened to scenario bounds.')
    except Exception as e:
        print(f'  Stage 2 WARNING: {e}')
        for vname, (lo, up) in stage2_prev_bounds.items():
            gekko_vars[vname].lower = lo; gekko_vars[vname].upper = up

    # ── STAGE 3: MINLP integer lock (APOPT) ────────────────────────────────────
    print('\n──── STAGE 3: MINLP integer lock (APOPT) ────')
    for vname, vd in var_defs.items():
        if vname not in gekko_vars: continue
        if vd['is_integer']:
            gekko_vars[vname].lower = int(round(vd['lb_phys']))
            gekko_vars[vname].upper = int(round(vd['ub_phys']))
        else:
            gekko_vars[vname].lower = vd['lb_phys']
            gekko_vars[vname].upper = vd['ub_phys']

    m.options.SOLVER = 1
    m.solver_options = [
        'minlp_maximum_iterations 5000',
        'minlp_max_iter_with_int_sol 1000',
        'minlp_branch_method 3',
        'minlp_integer_tol 0.1',
        'minlp_gap_tol 0.01',
        'nlp_maximum_iterations 1000',
    ]
    m.options.MAX_ITER = 5000
    SOLVE_SUCCESS = False
    try:
        m.solve(disp=True); SOLVE_SUCCESS = True
        print('\n>>> MINLP CONVERGED <<<')
    except Exception as e:
        print(f'\n>>> MINLP non-zero ({e}) — attempting recovery <<<')

    # Recovery: round integers, resolve as NLP
    if not SOLVE_SUCCESS:
        def _gval(v):
            try:
                r = v.value
                return float(list(r)[0] if hasattr(r, '__iter__') else r)
            except Exception: return None
        n_rounded = 0
        for vname, vd in var_defs.items():
            if not vd['is_integer'] or vname not in gekko_vars: continue
            cur = _gval(gekko_vars[vname])
            if cur is None: continue
            rounded = max(int(round(vd['lb_phys'])), min(int(round(vd['ub_phys'])), int(round(cur))))
            gekko_vars[vname].lower = rounded; gekko_vars[vname].upper = rounded; n_rounded += 1
        m.solver_options = []; m.options.SOLVER = 3; m.options.MAX_ITER = 500
        try:
            m.solve(disp=False); SOLVE_SUCCESS = True
            print(f'  Recovery NLP converged ({n_rounded} integers rounded)')
        except Exception as e:
            print(f'  Recovery also failed: {e}')

    solve_seconds = round(time.time() - t_solve, 1)
    print(f'\nSolve result: {"SUCCESS" if SOLVE_SUCCESS else "FAILED"}  ({solve_seconds}s)')

    # Infeasibilities report
    from pathlib import Path as _P
    _inf_path = _P(m._path) / 'infeasibilities.txt'
    infeas_text = None
    if _inf_path.exists():
        with open(str(_inf_path)) as _f: infeas_text = _f.read()
        print('\nINFEASIBILITIES REPORT:'); print(infeas_text[:2000])

    # ── RESULTS EXTRACTION + POST-PROCESSING ──────────────────────────────────

    def gval(v):
        try:
            r = v.value
            return float(list(r)[0] if hasattr(r, '__iter__') else r)
        except Exception: return float('nan')

    # ── 1. Extract optimizer results ───────────────────────────────────────────
    opt_vals = {}
    for vname, gv in gekko_vars.items():
        a = actual_ns.get(vname, np.nan)
        o = gval(gv)
        delta = (o - a) if not (np.isnan(a) or np.isnan(o)) else np.nan
        opt_vals[vname] = {'actual':a, 'optimum':o, 'delta':delta,
                           'is_integer':var_defs[vname]['is_integer']}


    opt_ctx = dict(actual_ns)
    for vname, vals in opt_vals.items():
        if not np.isnan(vals['optimum']): opt_ctx[vname] = vals['optimum']

    opt_derived = {}
    for tag in sorted_derived:
        f = derived_eq_map.get(tag,'')
        if not f: continue
        val = safe_eval_scalar(f, opt_ctx, all_referenced_tags)
        if not np.isnan(val): opt_ctx[tag] = val; opt_derived[tag] = val

    opt_inferred = {}
    opt_inf_ctx = dict(opt_ctx)
    # Apply scenario prices to optimum context too
    opt_inf_ctx.update(prices)
    opt_inf_ctx['Fuel_Cost_in_MMBTU'] = prices['Fuel_Rate'] * FUEL_COST_IN_MMBTU_BASELINE / BASELINE['Fuel_Rate']
    for tag in sorted_inf:
        f = inferred_formula_map.get(tag,'')
        if not f.strip(): continue
        v = safe_eval_scalar(f, opt_inf_ctx, all_referenced_tags)
        if not np.isnan(v): opt_inf_ctx[tag] = v; opt_inferred[tag] = v

    obj_actual  = actual_ns.get(obj_tag, np.nan)
    obj_optimum = opt_inf_ctx.get(obj_tag, np.nan)
    solver_obj  = np.nan
    if SOLVE_SUCCESS:
        try:
            solver_obj = m.options.OBJFCNVAL
            if obj_dir == 1: solver_obj = -solver_obj
        except Exception: pass

    # ── 2. Solve classification ────────────────────────────────────────────────
    def _nan(x): return x is None or (isinstance(x,float) and x!=x)
    baseline_obj = obj_actual if not _nan(obj_actual) else np.nan
    optimum_obj  = solver_obj if not _nan(solver_obj) else obj_optimum
    saving       = (baseline_obj - optimum_obj) if not (_nan(baseline_obj) or _nan(optimum_obj)) else np.nan
    saving_pct   = (saving / baseline_obj * 100) if (not _nan(saving) and baseline_obj and abs(baseline_obj)>1e-6) else 0.0

    if not SOLVE_SUCCESS or _nan(optimum_obj):
        solve_status = 'infeasible_no_optimum'
    elif bound_patches and not _nan(saving) and abs(saving) < 1e-3:
        solve_status = 'infeasible_no_optimum'
    elif infeas_text:
        solve_status = 'ok_with_warnings'
    else:
        solve_status = 'ok'

    print(f'  Solve status: {solve_status.upper()}')
    if not _nan(saving):
        print(f'  Baseline: ${baseline_obj:,.2f}/hr   Optimum: ${optimum_obj:,.2f}/hr')
        print(f'  Saving:   ${saving:,.2f}/hr  ({saving_pct:.2f}%)  Annual: ${saving*8760/1e6:.3f} M/yr')

    # ── 3. Switchovers ─────────────────────────────────────────────────────────
    switchovers = []
    for tag, vals in opt_vals.items():
        if not tag.endswith('_Status'): continue
        if _nan(vals['actual']) or _nan(vals['optimum']): continue
        a = int(round(vals['actual'])); o = int(round(vals['optimum']))
        if a != o:
            label = tag.replace('_Status','').replace('_',' ')
            switchovers.append({'tag':tag,'label':label,'actual':a,'optimum':o,
                                'direction':'ON' if o>a else 'OFF'})
    print(f'  Switchovers: {len(switchovers)}')
    for sw in switchovers: print(f'    {sw["label"]}: {sw["actual"]}→{sw["direction"]}')

    # ── 4. Build dual namespace for post-processing ────────────────────────────
    ns = {}
    all_base_tags = set(list(actual_ns.keys()) + list(opt_vals.keys()) +
                        list(opt_derived.keys()) + list(opt_inferred.keys()))
    for tag in all_base_tags:
        bare = tag.replace('_actual','').replace('_optimum','') if (tag.endswith('_actual') or tag.endswith('_optimum')) else tag
        a_val = actual_ns.get(bare, actual_ns.get(f'{bare}_actual', np.nan))
        if bare in opt_vals: o_val = opt_vals[bare]['optimum']
        elif bare in opt_derived: o_val = opt_derived[bare]
        elif bare in opt_inferred: o_val = opt_inferred[bare]
        else: o_val = a_val
        ns[f'{bare}_actual']  = a_val
        ns[f'{bare}_optimum'] = o_val
        ns[bare] = a_val

    # ── 5. Energy bills ────────────────────────────────────────────────────────
    power_bill_a = _safe_float(ns.get('Power_Bill_actual'))
    power_bill_o = _safe_float(ns.get('Power_Bill_optimum'))
    fuel_bill_a  = _safe_float(ns.get('Fuel_Bill_actual'))
    fuel_bill_o  = _safe_float(ns.get('Fuel_Bill_optimum'))
    dmw_bill_a   = _safe_float(ns.get('DMW_Bill_actual'))
    dmw_bill_o   = _safe_float(ns.get('DMW_Bill_optimum'))
    total_bill_a = power_bill_a + fuel_bill_a + dmw_bill_a
    total_bill_o = power_bill_o + fuel_bill_o + dmw_bill_o

    # ── 6. Process health ─────────────────────────────────────────────────────
    def _ph(key): return (_safe_float(ns.get(f'{key}_actual')), _safe_float(ns.get(f'{key}_optimum')))
    flare_a,   flare_o   = _ph('Flare_Total_Flow')
    vent_a,    vent_o    = _ph('Total_Steam_Vent')
    air_mot_a, air_mot_o = _ph('Total_Air_Compressor_Motor_Power')
    air_trb_a, air_trb_o = _ph('Total_Air_Compressor_Turbine_Steam')

    # ── 7. SEU evaluation (inlined) ───────────────────────────────────────────
    print('\n──── SEU EVALUATION ────')
    seu_df = pd.DataFrame()
    try:
        v7 = cfg['seu_detail'].copy()
        dbseu = db.get('seu_details', pd.DataFrame())
        dbseu_by_id = {}
        if not dbseu.empty:
            for _, dr in dbseu.iterrows():
                try: dbseu_by_id[int(dr['seu_id'])] = dr
                except Exception: pass
        seu_rows = []
        for _, r in v7.iterrows():
            sid = int(r['seu_id']); name = r.get('seu_name','')
            def ev(expr):
                if expr is None or (isinstance(expr,float) and np.isnan(expr)): return np.nan
                s = str(expr).strip()
                if not s or s.lower()=='nan': return np.nan
                return safe_eval_scalar(s, ns)
            dbrow = dbseu_by_id.get(sid)
            def pick(col):
                if dbrow is not None:
                    v = dbrow.get(col)
                    if v is not None and not (isinstance(v,float) and np.isnan(v)) and str(v).strip().lower()!='nan':
                        return v
                return r.get(col)
            baseline = ev(pick('baseline_duty_expression'))
            actual   = ev(pick('actual_duty_expression'))
            target   = ev(pick('target_duty_expression'))
            gain     = ev(pick('gain_expression'))
            enpi     = ev(pick('enpi_expression'))
            bfac     = ev(r.get('benefit_factor'))
            enpi_benefit = enpi*bfac if not (np.isnan(enpi) if isinstance(enpi,float) else False) and not (np.isnan(bfac) if isinstance(bfac,float) else True) else np.nan
            gain_benefit = gain*bfac if not (np.isnan(gain) if isinstance(gain,float) else False) and not (np.isnan(bfac) if isinstance(bfac,float) else True) else np.nan
            seu_rows.append(dict(seu_id=sid,seu_name=str(name),actual=actual,target=target,
                                 baseline=baseline,gain=gain,enpi=enpi,
                                 enpi_benefit=enpi_benefit,gain_benefit=gain_benefit,benefit_factor=bfac))
        seu_df = pd.DataFrame(seu_rows)
        total_enpi_benefit = float(pd.to_numeric(seu_df['enpi_benefit'],errors='coerce').fillna(0).sum())
        total_gain_benefit = float(pd.to_numeric(seu_df['gain_benefit'],errors='coerce').fillna(0).sum())
        print(f'  SEU rows: {len(seu_df)}  total enpi_benefit: ${total_enpi_benefit:.2f}/hr')
    except Exception as _e:
        total_enpi_benefit = 0.0; total_gain_benefit = 0.0
        print(f'  SEU evaluation failed: {_e}')

    equipment_switching_benefit = (_safe_float(saving) - total_enpi_benefit) if not _nan(saving) else 0.0

    # ── 8. SEEC KPIs (inlined) ────────────────────────────────────────────────
    seec_df = pd.DataFrame()
    seec_gain = seec_enpi = np.nan
    try:
        sk_rows = db.get('seec_kpi', pd.DataFrame())
        if not seu_df.empty and not sk_rows.empty:
            b  = float(pd.to_numeric(seu_df['baseline'],errors='coerce').fillna(0).sum())
            a  = float(pd.to_numeric(seu_df['actual'],  errors='coerce').fillna(0).sum())
            t  = float(pd.to_numeric(seu_df['target'],  errors='coerce').fillna(0).sum())
            g  = float(pd.to_numeric(seu_df['gain'],    errors='coerce').fillna(0).sum())
            ep = float(pd.to_numeric(seu_df['enpi'],    errors='coerce').fillna(0).sum())
            name_to_val = {'seec_baseline':b,'seec_actual':a,'seec_target':t,'seec_gain':g,'seec_enpi':ep}
            seec_rows = []
            for _, r in sk_rows.iterrows():
                nm = str(r['seec_kpi_name']).strip().lower()
                seec_rows.append({'seec_kpi_name':r['seec_kpi_name'],
                                  'seec_kpi_value':name_to_val.get(nm,np.nan)})
            seec_df = pd.DataFrame(seec_rows)
            seec_gain = name_to_val.get('seec_gain', np.nan)
            seec_enpi = name_to_val.get('seec_enpi', np.nan)
    except Exception as _e: print(f'  SEEC failed: {_e}')

    # ── 9. ODS (inlined, requires DB) ─────────────────────────────────────────
    ods_df = pd.DataFrame()
    try:
        dbods    = db.get('operation_decision_support', pd.DataFrame())
        dbcause  = db.get('cause', pd.DataFrame())
        dbeffect = db.get('effect', pd.DataFrame())
        if not (dbods.empty or dbcause.empty or dbeffect.empty):
            ods_sub = dbods[(dbods['model_id']==MODEL_ID) & (dbods['active']==1)].copy()
            cause_idx  = dbcause.set_index('cause_id')
            effect_idx = dbeffect.set_index('effect_id')
            v7cause  = cfg.get('cause', pd.DataFrame())
            v7effect = cfg.get('effect', pd.DataFrame())
            v7c_expr = dict(zip(v7cause['cause_name'].astype(str),  v7cause.get('cause_expression',  pd.Series([])).astype(str))) if not v7cause.empty else {}
            v7e_expr = dict(zip(v7effect['effect_name'].astype(str),v7effect.get('effect_expression',pd.Series([])).astype(str))) if not v7effect.empty else {}
            def _fires(expr):
                if not expr: return 0
                v = safe_eval_scalar(expr, ns)
                return 1 if not np.isnan(v) and float(v)!=0 else 0
            ods_rows = []
            for _, r in ods_sub.iterrows():
                cid=int(r['cause_id']); eid=int(r['effect_id'])
                if cid not in cause_idx.index or eid not in effect_idx.index: continue
                crow=cause_idx.loc[cid]; erow=effect_idx.loc[eid]
                cname=str(crow['cause_name']); ename=str(erow['effect_name'])
                c_expr = str(crow.get('cause_expression','')).strip() or v7c_expr.get(cname,'')
                e_expr = str(erow.get('effect_expression','')).strip() or v7e_expr.get(ename,'')
                if _fires(c_expr)==1 and _fires(e_expr)==1:
                    ods_rows.append({'cause_name':cname,'effect_name':ename,
                                     'cause_id':cid,'effect_id':eid})
            ods_df = pd.DataFrame(ods_rows)
        print(f'  ODS alert pairs: {len(ods_df)}')
    except Exception as _e: print(f'  ODS skipped: {_e}')

    # ── 10. PEEO (inlined) ────────────────────────────────────────────────────
    peeo_df = pd.DataFrame()
    try:
        peeo_sheet = cfg.get('peeo_based_adjustment', pd.DataFrame())
        if not peeo_sheet.empty:
            parent_col = next((c for c in ['parent_tag_name','parent_tag','parent'] if c in peeo_sheet.columns), None)
            step_col   = next((c for c in ['step_change_value','step_change','step'] if c in peeo_sheet.columns), None)
            type_col   = next((c for c in ['suggestion_type','type'] if c in peeo_sheet.columns), None)
            peeo_rows = []
            for _, r in peeo_sheet.iterrows():
                parent = str(r.get(parent_col,'')) if parent_col else ''
                stype  = str(r.get(type_col,'')).strip().lower() if type_col else ''
                a = ns.get(parent+'_actual', np.nan); o = ns.get(parent+'_optimum', np.nan)
                if not (np.isnan(a) or np.isnan(o)) and stype=='yes' and abs(float(a)-float(o))>1e-4:
                    peeo_rows.append({'parent_tag':parent,'parent_actual':a,'parent_optimum':o,
                                      'step_change_value':r.get(step_col) if step_col else np.nan})
            peeo_df = pd.DataFrame(peeo_rows)
        print(f'  PEEO triggered: {len(peeo_df)}')
    except Exception as _e: print(f'  PEEO skipped: {_e}')

    # ── 11. KEVs ──────────────────────────────────────────────────────────────
    kev_rows = []
    for tag in ns:
        if tag.endswith('_KEV_actual') or (tag.endswith('_KEV') and f'{tag}_actual' not in ns):
            base = tag.replace('_actual','')
            a = _safe_float(ns.get(f'{base}_actual')); o = _safe_float(ns.get(f'{base}_optimum'))
            kev_rows.append({'tag':base,'actual':a,'optimum':o,'delta':o-a})
    print(f'  KEV rows: {len(kev_rows)}')

    # ── SENSITIVITY SWEEP (frozen-topology approximation) ─────────────────────
    # Runs only when RUN_SENSITIVITY_SWEEP = True (set in CONFIG cell).
    # Uses the optimum drive topology found by GEKKO for the main scenario and
    # re-evaluates the objective at ±15% price levels — fast, no GEKKO re-run.

    sensitivity_results = []

    if RUN_SENSITIVITY_SWEEP and solve_status in ('ok','ok_with_warnings'):
        print('Running 9-point sensitivity sweep (frozen topology)...')

        _levers = {'Power_Rate': prices['Power_Rate'], 'Fuel_Rate': prices['Fuel_Rate'], 'DMW_Rate': prices['DMW_Rate']}
        _multipliers = [0.85, 1.00, 1.15]

        def _eval_obj_at_prices(p_rate, f_rate, d_rate, base_ctx):
            ctx = dict(base_ctx)
            ctx['Power_Rate'] = p_rate; ctx['Fuel_Rate'] = f_rate; ctx['DMW_Rate'] = d_rate
            ctx['Fuel_Cost_in_MMBTU'] = f_rate * FUEL_COST_IN_MMBTU_BASELINE / BASELINE['Fuel_Rate']
            for tag in sorted_inf:
                frm = inferred_formula_map.get(tag,'')
                if frm.strip():
                    v = safe_eval_scalar(frm, ctx, all_referenced_tags)
                    if not np.isnan(v): ctx[tag] = v
            ctx.update({'Power_Rate':p_rate,'Fuel_Rate':f_rate,'DMW_Rate':d_rate})
            return _safe_float(ctx.get(obj_tag), np.nan)

        for lever, base_price in _levers.items():
            for mult in _multipliers:
                sp = dict(prices)
                sp[lever] = round(base_price * mult, 6)
                p = sp['Power_Rate']; f = sp['Fuel_Rate']; d = sp['DMW_Rate']
                bl = _eval_obj_at_prices(p, f, d, actual_ns)
                op = _eval_obj_at_prices(p, f, d, {**actual_ns, **{k:opt_vals[k]['optimum'] for k in opt_vals if not np.isnan(opt_vals[k]['optimum'])}})
                sv = bl - op
                sensitivity_results.append({
                    'lever': lever, 'multiplier': mult, 'price': round(sp[lever],4),
                    'baseline_obj': bl, 'optimum_obj': op, 'saving': sv,
                    'saving_pct': sv/bl*100 if abs(bl)>1e-6 else 0.0,
                })
                print(f'  {lever} x{mult:.2f}: baseline=${bl:,.0f} saving=${sv:,.0f}')

        # Tornado data
        tornado = []
        for lever in _levers:
            rows = [r for r in sensitivity_results if r['lever']==lever]
            low_sv  = next((r['saving'] for r in rows if r['multiplier']==0.85), np.nan)
            base_sv = next((r['saving'] for r in rows if r['multiplier']==1.00), np.nan)
            high_sv = next((r['saving'] for r in rows if r['multiplier']==1.15), np.nan)
            tornado.append({'lever':lever,'low_saving':low_sv,'base_saving':base_sv,'high_saving':high_sv,
                            'range': abs(high_sv-low_sv) if not np.isnan(high_sv) and not np.isnan(low_sv) else 0})
        tornado.sort(key=lambda x: x['range'], reverse=True)
        print('\nTornado (most sensitive first):')
        for t in tornado: print(f'  {t["lever"]:<15}: low=${t["low_saving"]:,.0f}  base=${t["base_saving"]:,.0f}  high=${t["high_saving"]:,.0f}  range=${t["range"]:,.0f}')
    else:
        tornado = []
        if not RUN_SENSITIVITY_SWEEP:
            print('Sensitivity sweep skipped (RUN_SENSITIVITY_SWEEP=False). Set True in CONFIG to run.')
        else:
            print('Sensitivity sweep skipped (solve did not converge).')

    # ── GENERATE EXCEL REPORT ─────────────────────────────────────────────────
    import xlsxwriter

    _ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    report_name = f'eo_report_{SCENARIO}_{_ts}'
    excel_path  = ROOT / f'{report_name}.xlsx'

    wb = xlsxwriter.Workbook(str(excel_path))

    # ── Formats ────────────────────────────────────────────────────────────────
    def _f(d): return wb.add_format(d)
    FMT = {
        'title':   _f({'bold':True,'font_size':16,'font_color':'#1F3864'}),
        'h2':      _f({'bold':True,'font_size':11,'bg_color':'#2F75B6','font_color':'white','border':1}),
        'header':  _f({'bold':True,'bg_color':'#D6E4F0','border':1,'align':'center'}),
        'label':   _f({'bold':True,'bg_color':'#F2F2F2','border':1}),
        'value':   _f({'border':1,'num_format':'#,##0.00'}),
        'value4':  _f({'border':1,'num_format':'#,##0.0000'}),
        'pct':     _f({'border':1,'num_format':'0.00%'}),
        'int':     _f({'border':1,'num_format':'0'}),
        'text':    _f({'border':1}),
        'center':  _f({'border':1,'align':'center'}),
        'green':   _f({'bold':True,'bg_color':'#E2EFDA','font_color':'#375623','border':1,'num_format':'#,##0.00'}),
        'red':     _f({'bold':True,'bg_color':'#FCE4D6','font_color':'#9C0006','border':1,'num_format':'#,##0.00'}),
        'green_bg':_f({'bold':True,'bg_color':'#E2EFDA','font_color':'#375623','border':2,'font_size':14,'align':'center','valign':'vcenter'}),
        'red_bg':  _f({'bold':True,'bg_color':'#FCE4D6','font_color':'#9C0006','border':2,'font_size':14,'align':'center','valign':'vcenter'}),
        'on':      _f({'bold':True,'bg_color':'#E2EFDA','font_color':'#375623','border':1,'align':'center'}),
        'off':     _f({'bold':True,'bg_color':'#FCE4D6','font_color':'#9C0006','border':1,'align':'center'}),
        'note':    _f({'italic':True,'font_color':'#595959','font_size':9}),
        'warn':    _f({'italic':True,'font_color':'#9C6500','font_size':10,'bg_color':'#FFEB9C','border':1,'text_wrap':True}),
        'subtitle':_f({'bold':True,'font_size':11,'font_color':'#2F75B6'}),
    }

    def _hdr(ws, row, cols):
        for c, label in enumerate(cols): ws.write(row, c, label, FMT['header'])

    def _v(val, default=0.0):
        if val is None: return default
        try: f=float(val); return default if f!=f else f
        except: return default

    def _sw(ws, row, col, val, fmt):
        if val is None or (isinstance(val,float) and val!=val): ws.write(row,col,'',FMT['text'])
        else: ws.write(row,col,val,fmt)

    # ═══ SHEET 1 — SUMMARY ════════════════════════════════════════════════════
    ws1 = wb.add_worksheet('Summary')
    ws1.set_column(0,0,32); ws1.set_column(1,1,22); ws1.set_column(2,2,14); ws1.set_column(3,3,22)
    ws1.set_row(3,40)

    ws1.write(0,0,'Energy Optimization — Report', FMT['title'])
    ws1.write(1,0,f'Scenario: {SCENARIO}', FMT['subtitle'])
    ws1.write(2,0,f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}', FMT['note'])

    _status_fmt = FMT['green_bg'] if solve_status in ('ok','ok_with_warnings') else FMT['red_bg']
    _status_txt = 'OPTIMUM FOUND' if solve_status in ('ok','ok_with_warnings') else 'OPTIMUM NOT FOUND'
    ws1.merge_range(3,0,3,3, _status_txt, _status_fmt)

    r = 5
    ws1.write(r,0,'Prices',FMT['h2']); ws1.write(r,1,'Value',FMT['h2']); ws1.write(r,2,'Unit',FMT['h2']); r+=1
    for label,val,unit in [('Power Rate',prices['Power_Rate'],'$/MWh'),
                            ('Fuel Rate', prices['Fuel_Rate'], '$/MMBTU'),
                            ('DMW Rate',  prices['DMW_Rate'],  '$/t')]:
        ws1.write(r,0,label,FMT['label']); _sw(ws1,r,1,val,FMT['value4']); ws1.write(r,2,unit,FMT['text']); r+=1

    r+=1
    ws1.write(r,0,'Optimization Results',FMT['h2']); ws1.write(r,1,'Actual',FMT['h2']); ws1.write(r,2,'Optimum',FMT['h2']); ws1.write(r,3,'Saving',FMT['h2']); r+=1
    for label,a_val,o_val in [
        ('Objective ($/hr)',    baseline_obj, optimum_obj),
        ('Power Bill ($/hr)',   power_bill_a,  power_bill_o),
        ('Fuel Bill ($/hr)',    fuel_bill_a,   fuel_bill_o),
        ('DMW Bill ($/hr)',     dmw_bill_a,    dmw_bill_o),
    ]:
        sv = _v(a_val) - _v(o_val)
        ws1.write(r,0,label,FMT['label'])
        _sw(ws1,r,1,a_val,FMT['value']); _sw(ws1,r,2,o_val,FMT['value'])
        _sw(ws1,r,3,sv, FMT['green'] if sv>=0 else FMT['red']); r+=1

    r+=1
    ws1.write(r,0,'Saving ($/hr)',FMT['label'])
    _sw(ws1,r,1,saving,FMT['green'] if _v(saving)>=0 else FMT['red']); r+=1
    ws1.write(r,0,'Saving (%)',FMT['label']); _sw(ws1,r,1,saving_pct/100 if saving_pct else 0,FMT['pct']); r+=1
    ws1.write(r,0,'Annual Saving ($M/yr)',FMT['label']); _sw(ws1,r,1,_v(saving)*8760/1e6,FMT['value4']); r+=1
    ws1.write(r,0,'Equip. Switching Benefit ($/hr)',FMT['label']); _sw(ws1,r,1,equipment_switching_benefit,FMT['value']); r+=1
    ws1.write(r,0,'SEU Setpoint Benefit ($/hr)',FMT['label']); _sw(ws1,r,1,total_enpi_benefit,FMT['value']); r+=1
    ws1.write(r,0,'Solve Time (s)',FMT['label']); ws1.write(r,1,solve_seconds,FMT['int']); r+=2

    ws1.write(r,0,'Top Switchovers',FMT['h2']); ws1.write(r,1,'Actual',FMT['h2']); ws1.write(r,2,'Optimum',FMT['h2']); ws1.write(r,3,'Direction',FMT['h2']); r+=1
    for sw in switchovers[:10]:
        ws1.write(r,0,sw['label'],FMT['text']); ws1.write(r,1,sw['actual'],FMT['int']); ws1.write(r,2,sw['optimum'],FMT['int'])
        ws1.write(r,3,sw['direction'], FMT['on'] if sw['direction']=='ON' else FMT['off']); r+=1

    # ═══ SHEET 2 — SWITCHOVERS ════════════════════════════════════════════════
    ws2 = wb.add_worksheet('Switchovers')
    ws2.set_column(0,0,40); ws2.set_column(1,3,12); ws2.set_column(4,4,14)
    ws2.write(0,0,'Equipment Switchovers',FMT['title']); r=2
    _hdr(ws2,r,['Equipment','Tag','Actual','Optimum','Direction']); r+=1
    for sw in switchovers:
        ws2.write(r,0,sw['label'],FMT['text']); ws2.write(r,1,sw['tag'],FMT['text'])
        ws2.write(r,2,sw['actual'],FMT['int']); ws2.write(r,3,sw['optimum'],FMT['int'])
        ws2.write(r,4,sw['direction'], FMT['on'] if sw['direction']=='ON' else FMT['off']); r+=1
    if not switchovers: ws2.write(r,0,'No switchovers',FMT['note'])

    # ═══ SHEET 3 — SEU DRILL-DOWN ═════════════════════════════════════════════
    ws3 = wb.add_worksheet('SEU Drill-Down')
    ws3.set_column(0,0,8); ws3.set_column(1,1,35); ws3.set_column(2,8,14)
    ws3.write(0,0,'SEU Drill-Down',FMT['title'])
    ws3.write(1,0,f'Total ENPI Benefit: ${total_enpi_benefit:.2f}/hr', FMT['subtitle'])
    r=3
    if not seu_df.empty:
        _hdr(ws3,r,['ID','SEU Name','Actual','Target','Baseline','Gain','ENPI','ENPI Benefit $/hr','Gain Benefit $/hr']); r+=1
        for _, row in seu_df.iterrows():
            ws3.write(r,0,row['seu_id'],FMT['int']); ws3.write(r,1,str(row['seu_name']),FMT['text'])
            for ci, col in enumerate(['actual','target','baseline','gain','enpi','enpi_benefit','gain_benefit']):
                v = _v(row.get(col), np.nan)
                if not np.isnan(v): ws3.write(r,2+ci,v,FMT['value'])
                else: ws3.write(r,2+ci,'',FMT['text'])
            r+=1
        ws3.write(r+1,0,'Note: SEU ENPI benefit measures setpoint efficiency only.',FMT['note'])
        ws3.write(r+2,0,'Equipment topology switching benefit is reported separately on Summary sheet.',FMT['note'])
    else:
        ws3.write(r,0,'SEU data not available',FMT['note'])

    # ═══ SHEET 4 — OPTIMIZATION KPIs ═════════════════════════════════════════
    ws4 = wb.add_worksheet('Optimization KPIs')
    ws4.set_column(0,0,38); ws4.set_column(1,2,18)
    ws4.write(0,0,'Optimization KPIs',FMT['title']); r=2
    _hdr(ws4,r,['KPI','Actual','Optimum']); r+=1
    kpi_rows = [
        ('Objective ($/hr)',            baseline_obj, optimum_obj),
        ('Power Bill ($/hr)',           power_bill_a, power_bill_o),
        ('Fuel Bill ($/hr)',            fuel_bill_a,  fuel_bill_o),
        ('DMW Bill ($/hr)',             dmw_bill_a,   dmw_bill_o),
        ('Total Energy Bill ($/hr)',    total_bill_a, total_bill_o),
        ('Saving ($/hr)',               None,         saving),
        ('Saving (%)',                  None,         saving_pct),
        ('Annual Saving ($M/yr)',       None,         _v(saving)*8760/1e6),
        ('Equip Switch Benefit ($/hr)', None,         equipment_switching_benefit),
        ('SEU ENPI Benefit ($/hr)',     None,         total_enpi_benefit),
        ('SEEC Gain',                   None,         seec_gain),
        ('SEEC ENPI',                   None,         seec_enpi),
        ('Flare Flow (t/hr)',           flare_a,      flare_o),
        ('Steam Vent (t/hr)',           vent_a,       vent_o),
        ('Air Comp Motor Power (kW)',   air_mot_a,    air_mot_o),
        ('Air Comp Turbine Steam (t/hr)',air_trb_a,   air_trb_o),
    ]
    for label, av, ov in kpi_rows:
        ws4.write(r,0,label,FMT['label'])
        _sw(ws4,r,1,av,FMT['value']); _sw(ws4,r,2,ov,FMT['value']); r+=1

    # ═══ SHEET 5 — SENSITIVITY ════════════════════════════════════════════════
    ws5 = wb.add_worksheet('Sensitivity Analysis')
    ws5.set_column(0,0,20); ws5.set_column(1,6,14)
    ws5.write(0,0,'Sensitivity Analysis (Frozen-Topology Approximation)',FMT['title'])
    r=2
    if sensitivity_results:
        _hdr(ws5,r,['Lever','Multiplier','Price','Baseline $/hr','Optimum $/hr','Saving $/hr','Saving %']); r+=1
        for row in sensitivity_results:
            ws5.write(r,0,row['lever'],FMT['text']); _sw(ws5,r,1,row['multiplier'],FMT['value'])
            _sw(ws5,r,2,row['price'],FMT['value4']); _sw(ws5,r,3,row['baseline_obj'],FMT['value'])
            _sw(ws5,r,4,row['optimum_obj'],FMT['value']); _sw(ws5,r,5,row['saving'],FMT['value'])
            _sw(ws5,r,6,row['saving_pct']/100 if row['saving_pct'] else 0,FMT['pct']); r+=1
        r+=2
        ws5.write(r,0,'Tornado (ranked by range)',FMT['h2'])
        ws5.write(r,1,'Low Saving',FMT['h2']); ws5.write(r,2,'Base Saving',FMT['h2'])
        ws5.write(r,3,'High Saving',FMT['h2']); ws5.write(r,4,'Range',FMT['h2']); r+=1
        for t in tornado:
            ws5.write(r,0,t['lever'],FMT['label'])
            _sw(ws5,r,1,t['low_saving'],FMT['value']); _sw(ws5,r,2,t['base_saving'],FMT['value'])
            _sw(ws5,r,3,t['high_saving'],FMT['value']); _sw(ws5,r,4,t['range'],FMT['value']); r+=1
    else:
        ws5.write(r,0,'Set RUN_SENSITIVITY_SWEEP = True in CONFIG cell to generate this analysis.',FMT['warn'])

    # ═══ SHEET 6 — ODS & PEEO ════════════════════════════════════════════════
    ws6 = wb.add_worksheet('ODS & PEEO')
    ws6.set_column(0,0,40); ws6.set_column(1,3,30)
    ws6.write(0,0,'Operation Decision Support (ODS)',FMT['title']); r=2
    if not ods_df.empty:
        _hdr(ws6,r,['Cause','Effect','Cause ID','Effect ID']); r+=1
        for _, row in ods_df.iterrows():
            ws6.write(r,0,str(row.get('cause_name','')),FMT['text'])
            ws6.write(r,1,str(row.get('effect_name','')),FMT['text'])
            _sw(ws6,r,2,row.get('cause_id'),FMT['int']); _sw(ws6,r,3,row.get('effect_id'),FMT['int']); r+=1
    else:
        ws6.write(r,0,'No ODS alerts fired (or DB CSV files not found).',FMT['note']); r+=1

    r+=2; ws6.write(r,0,'PEEO Step-Change Suggestions',FMT['title']); r+=2
    if not peeo_df.empty:
        _hdr(ws6,r,['Parent Tag','Actual','Optimum','Step Change Value']); r+=1
        for _, row in peeo_df.iterrows():
            ws6.write(r,0,str(row.get('parent_tag','')),FMT['text'])
            _sw(ws6,r,1,row.get('parent_actual'),FMT['value']); _sw(ws6,r,2,row.get('parent_optimum'),FMT['value'])
            _sw(ws6,r,3,row.get('step_change_value'),FMT['value']); r+=1
    else:
        ws6.write(r,0,'No PEEO suggestions triggered.',FMT['note'])

    # ═══ SHEET 7 — DATA QUALITY ══════════════════════════════════════════════
    ws7 = wb.add_worksheet('Data Quality')
    ws7.set_column(0,0,42); ws7.set_column(1,1,22)
    ws7.write(0,0,'Data Quality & Methodology Notes',FMT['title']); r=2

    _dq = [
        ('Scenario',                          SCENARIO),
        ('Solve Status',                      solve_status),
        ('Solve Time (s)',                     str(solve_seconds)),
        ('MINLP gap_tol',                      '0.01'),
        ('Decision Variables',                 str(len(var_defs))),
        ('Integer Variables',                  str(sum(1 for v in var_defs.values() if v['is_integer']))),
        ('Switchovers',                        str(len(switchovers))),
        ('SEU Rows',                           str(len(seu_df))),
        ('SEU Total ENPI Benefit ($/hr)',      f'{total_enpi_benefit:.2f}'),
        ('Equipment Switching Benefit ($/hr)', f'{equipment_switching_benefit:.2f}'),
        ('Bound Patches Applied',              str(len(bound_patches))),
        ('Note: SEU decomposition',           'ENPI benefit measures setpoint efficiency only. '
                                              'Does not include equipment topology switching benefit.'),
        ('Note: CW drive statuses',           'Locked at PI values (FIX-3) — no CW constraint in Excel.'),
        ('Note: DMW/BFW physics constraints', 'Injected in Python (FIX-2) — Excel rows are dormant (*0 bug).'),
        ('Note: Integer flag sanitation',     'Applied in Python (FIX-1) — Excel flags contain errors.'),
    ]
    _hdr(ws7, r, ['Item','Value']); r+=1
    for item, val in _dq:
        fmt_v = FMT['warn'] if item.startswith('Note') else FMT['text']
        ws7.write(r,0,item,FMT['label']); ws7.write(r,1,val,fmt_v); r+=1
    if infeas_text:
        r+=1; ws7.write(r,0,'Infeasibility Report',FMT['h2']); r+=1
        for line in (infeas_text[:2000]).split('\\n'):
            ws7.write(r,0,line,FMT['note']); r+=1

    wb.close()
    print(f'\\nExcel report saved: {excel_path.name}')

    pdf_path = None  # PDF generation disabled (source cell was broken/truncated)

    # ── Build summary dict for combined report ───────────────────────────────
    return {
        'scenario':        SCENARIO,
        'description':     _cfg.get('description', ''),
        'power_rate':      Power_Rate,
        'fuel_rate':       Fuel_Rate,
        'dmw_rate':        DMW_Rate,
        'n_bound_patches': len(bound_patches),
        'solve_status':    solve_status,
        'solve_seconds':   solve_seconds,
        'baseline_obj':    _v(baseline_obj, float('nan')),
        'optimum_obj':     _v(optimum_obj,  float('nan')),
        'saving':          _v(saving,        float('nan')),
        'saving_pct':      _v(saving_pct,    float('nan')),
        'annual_saving_M': _v(saving, 0.0) * 8760 / 1e6,
        'power_bill_a':    power_bill_a, 'power_bill_o': power_bill_o,
        'fuel_bill_a':     fuel_bill_a,  'fuel_bill_o':  fuel_bill_o,
        'dmw_bill_a':      dmw_bill_a,   'dmw_bill_o':   dmw_bill_o,
        'flare_a':         flare_a,      'flare_o':      flare_o,
        'vent_a':          vent_a,       'vent_o':       vent_o,
        'n_switchovers':   len(switchovers),
        'switchovers':     switchovers,
        'enpi_benefit':    total_enpi_benefit,
        'switch_benefit':  equipment_switching_benefit,
        'seec_gain':       seec_gain,
        'seec_enpi':       seec_enpi,
        'excel_path':      str(excel_path),
        'pdf_path':        str(pdf_path) if 'pdf_path' in dir() and pdf_path else None,
    }


In [ ]:
# ── RUN ALL SCENARIOS + WRITE COMBINED REPORT ────────────────────────────────
import xlsxwriter
from datetime import datetime
from pathlib import Path
import traceback

reports_dir = ROOT / REPORTS_DIR_NAME
reports_dir.mkdir(exist_ok=True)

results = []
errors  = []
t_run0  = time.time()

for SCENARIO in SCENARIOS_TO_RUN:
    t_s = time.time()
    try:
        res = run_scenario(SCENARIO)
        # Move per-scenario xlsx/pdf into reports_dir for tidy collection
        for k in ('excel_path', 'pdf_path'):
            p = res.get(k)
            if p and Path(p).exists():
                dest = reports_dir / Path(p).name
                try:
                    if Path(p).resolve() != dest.resolve():
                        shutil.move(p, dest)
                        res[k] = str(dest)
                except Exception:
                    pass
        results.append(res)
        print(f'\n>>> {SCENARIO}: {res["solve_status"].upper()} in {time.time()-t_s:.0f}s  '
              f'saving=${res["saving"]:.2f}/hr ({res["saving_pct"]:.2f}%)')
    except Exception as e:
        tb = traceback.format_exc()
        errors.append({'scenario': SCENARIO, 'error': str(e), 'traceback': tb})
        print(f'\n>>> {SCENARIO}: FAILED — {e}')
        if not CONTINUE_ON_ERROR:
            raise
        # Keep an error placeholder in the combined sheet
        results.append({'scenario': SCENARIO, 'solve_status': 'error',
                        'description': SCENARIO_CONFIGS[SCENARIO].get('description',''),
                        'error': str(e)})

print(f'\n\nTotal wall time: {(time.time()-t_run0)/60:.1f} min  |  '
      f'OK: {sum(1 for r in results if r.get("solve_status","").startswith("ok"))}/{len(SCENARIOS_TO_RUN)}')

# ── Combined cross-scenario report ───────────────────────────────────────────
_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
combined_path = ROOT / f'eo_all_scenarios_{_ts}.xlsx'
wb = xlsxwriter.Workbook(str(combined_path))
def _f(d): return wb.add_format(d)
F = {
    'title':  _f({'bold': True, 'font_size': 16, 'font_color': '#1F3864'}),
    'h2':     _f({'bold': True, 'bg_color': '#2F75B6', 'font_color': 'white', 'border': 1}),
    'header': _f({'bold': True, 'bg_color': '#D6E4F0', 'border': 1, 'align': 'center'}),
    'label':  _f({'bold': True, 'bg_color': '#F2F2F2', 'border': 1}),
    'text':   _f({'border': 1}),
    'value':  _f({'border': 1, 'num_format': '#,##0.00'}),
    'value4': _f({'border': 1, 'num_format': '#,##0.0000'}),
    'pct':    _f({'border': 1, 'num_format': '0.00%'}),
    'int':    _f({'border': 1, 'num_format': '0'}),
    'green':  _f({'bold': True, 'bg_color': '#E2EFDA', 'font_color': '#375623', 'border': 1, 'num_format': '#,##0.00'}),
    'red':    _f({'bold': True, 'bg_color': '#FCE4D6', 'font_color': '#9C0006', 'border': 1, 'num_format': '#,##0.00'}),
    'note':   _f({'italic': True, 'font_color': '#595959', 'font_size': 9}),
}

# ── Sheet 1 — Summary table (one row per scenario) ──────────────────────────
ws = wb.add_worksheet('Summary')
ws.set_column(0, 0, 20); ws.set_column(1, 1, 28); ws.set_column(2, 14, 14)
ws.write(0, 0, 'Energy Optimization — All-Scenarios Summary', F['title'])
ws.write(1, 0, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}', F['note'])
ws.write(2, 0, f'Scenarios run: {len(results)}  |  OK: {sum(1 for r in results if str(r.get("solve_status","")).startswith("ok"))}', F['note'])

cols = ['Scenario', 'Description', 'Status', 'Power', 'Fuel', 'DMW',
        'Baseline $/hr', 'Optimum $/hr', 'Saving $/hr', 'Saving %', 'Annual $M/yr',
        'Switchovers', 'ENPI Benefit', 'Switch Benefit', 'Solve sec']
for c, h in enumerate(cols): ws.write(4, c, h, F['header'])

r = 5
for res in results:
    ws.write(r, 0, res.get('scenario',''), F['label'])
    ws.write(r, 1, res.get('description',''), F['text'])
    status = str(res.get('solve_status','?'))
    fmt_st = F['green'] if status.startswith('ok') else F['red']
    ws.write(r, 2, status, fmt_st)
    def _wn(col, key, fmt):
        v = res.get(key)
        if v is None or (isinstance(v,float) and v != v):
            ws.write(r, col, '', F['text'])
        else:
            ws.write(r, col, float(v), fmt)
    _wn(3, 'power_rate', F['value4'])
    _wn(4, 'fuel_rate',  F['value4'])
    _wn(5, 'dmw_rate',   F['value4'])
    _wn(6, 'baseline_obj', F['value'])
    _wn(7, 'optimum_obj',  F['value'])
    sv = res.get('saving')
    if sv is None or (isinstance(sv,float) and sv != sv):
        ws.write(r, 8, '', F['text'])
    else:
        ws.write(r, 8, float(sv), F['green'] if float(sv) >= 0 else F['red'])
    spct = res.get('saving_pct')
    if spct is None or (isinstance(spct,float) and spct != spct):
        ws.write(r, 9, '', F['text'])
    else:
        ws.write(r, 9, float(spct)/100.0, F['pct'])
    _wn(10, 'annual_saving_M', F['value4'])
    _wn(11, 'n_switchovers',   F['int'])
    _wn(12, 'enpi_benefit',    F['value'])
    _wn(13, 'switch_benefit',  F['value'])
    _wn(14, 'solve_seconds',   F['value'])
    r += 1

# ── Sheet 2 — Bills breakdown ───────────────────────────────────────────────
ws2 = wb.add_worksheet('Bills')
ws2.set_column(0, 0, 20); ws2.set_column(1, 12, 14)
ws2.write(0, 0, 'Energy Bills by Scenario ($/hr)', F['title'])
hdr = ['Scenario', 'Power Actual','Power Optimum','Fuel Actual','Fuel Optimum',
       'DMW Actual','DMW Optimum','Total Actual','Total Optimum','Total Saving']
for c, h in enumerate(hdr): ws2.write(2, c, h, F['header'])
r = 3
for res in results:
    if not str(res.get('solve_status','')).startswith('ok'):
        ws2.write(r, 0, res.get('scenario',''), F['label']); r += 1; continue
    ws2.write(r, 0, res.get('scenario',''), F['label'])
    pa = float(res.get('power_bill_a') or 0); po = float(res.get('power_bill_o') or 0)
    fa = float(res.get('fuel_bill_a')  or 0); fo = float(res.get('fuel_bill_o')  or 0)
    da = float(res.get('dmw_bill_a')   or 0); do = float(res.get('dmw_bill_o')   or 0)
    ta, to = pa+fa+da, po+fo+do
    for c, v in enumerate([pa, po, fa, fo, da, do, ta, to], start=1):
        ws2.write(r, c, v, F['value'])
    sv = ta - to
    ws2.write(r, 9, sv, F['green'] if sv >= 0 else F['red'])
    r += 1

# ── Sheet 3 — Switchover map (scenarios across columns, equipment down rows) ─
ws3 = wb.add_worksheet('Switchovers')
ws3.set_column(0, 0, 38)
ws3.write(0, 0, 'Switchover Direction by Scenario', F['title'])
all_tags = []
seen = set()
for res in results:
    for sw in (res.get('switchovers') or []):
        if sw['tag'] not in seen:
            all_tags.append(sw); seen.add(sw['tag'])
scen_names = [r['scenario'] for r in results]
ws3.write(2, 0, 'Equipment', F['header'])
for c, sn in enumerate(scen_names, start=1):
    ws3.set_column(c, c, 14)
    ws3.write(2, c, sn, F['header'])
on_fmt  = _f({'bold': True, 'bg_color': '#E2EFDA', 'font_color': '#375623', 'border': 1, 'align': 'center'})
off_fmt = _f({'bold': True, 'bg_color': '#FCE4D6', 'font_color': '#9C0006', 'border': 1, 'align': 'center'})
neut    = _f({'border': 1, 'align': 'center', 'font_color': '#A6A6A6'})
for ri, sw in enumerate(all_tags, start=3):
    ws3.write(ri, 0, sw['label'], F['label'])
    for c, res in enumerate(results, start=1):
        match = next((s for s in (res.get('switchovers') or []) if s['tag'] == sw['tag']), None)
        if match: ws3.write(ri, c, match['direction'], on_fmt if match['direction']=='ON' else off_fmt)
        else: ws3.write(ri, c, '—', neut)
if not all_tags: ws3.write(3, 0, 'No switchovers across any scenario', F['note'])

# ── Sheet 4 — Errors (if any) ───────────────────────────────────────────────
if errors:
    ws4 = wb.add_worksheet('Errors')
    ws4.set_column(0, 0, 20); ws4.set_column(1, 1, 80)
    ws4.write(0, 0, 'Scenario Errors', F['title'])
    ws4.write(2, 0, 'Scenario', F['header']); ws4.write(2, 1, 'Error', F['header'])
    for ri, e in enumerate(errors, start=3):
        ws4.write(ri, 0, e['scenario'], F['label']); ws4.write(ri, 1, e['error'], F['text'])

wb.close()
print(f'\nCombined report: {combined_path.name}')
print(f'Per-scenario reports: {reports_dir}/')


In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────────────────────
print('=' * 72)
print('  ALL-SCENARIOS RUN — DONE')
print('=' * 72)
for res in results:
    s = str(res.get('solve_status','?')).upper()
    sv = res.get('saving')
    sv_s = f'${float(sv):,.2f}/hr' if sv is not None and not (isinstance(sv,float) and sv!=sv) else 'n/a'
    print(f'  {res["scenario"]:<20} {s:<25} saving={sv_s}')
print('=' * 72)
print(f'  Combined: {combined_path.name}')
print(f'  Folder:   {reports_dir}/')
print('=' * 72)
